# GemFinder — Empirical 95% Ranges for Model Outputs

## Purpose

GemFinder's neural network returns two outputs:

- **Shape** (`alpha`)
- **Rate** (`beta`)

This notebook measures how much these outputs vary when the unreliability matrix `rho` is slightly perturbed.

For each perturbation level, we calculate the empirical 95% sensitivity range:

`[2.5th percentile, 97.5th percentile]`

This is best described as an **empirical sensitivity / uncertainty range**, not a classical confidence interval.

## 1. Assumptions

Run this notebook after your main rho-sensitivity notebook.

It assumes these already exist:

- `X_counts`
- `Y_counts`
- `rho`
- `perturb_rho(...)`
- `predict_gamma_quiet(...)`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng_ci = np.random.default_rng(42)

## 2. Baseline outputs

Calculate the original GemFinder outputs using the unmodified unreliability matrix.

In [ ]:
baseline = predict_gamma_quiet(X_counts, Y_counts, rho)

baseline_alpha = baseline["alpha"]
baseline_beta = baseline["beta"]

print("Baseline alpha / shape:", baseline_alpha)
print("Baseline beta / rate  :", baseline_beta)

## 3. Repeated rho perturbations

For each perturbation size:

1. perturb `rho`;
2. keep the student response counts fixed;
3. run GemFinder;
4. record `alpha` and `beta`.

For a short first analysis, use 500 repetitions per perturbation level.

In [ ]:
SIGMAS = [0.001, 0.005, 0.01, 0.02]
N_REPETITIONS = 500

records = []

for sigma in SIGMAS:
    for rep in range(N_REPETITIONS):

        rho_p = perturb_rho(rho, sigma=sigma, rng=rng_ci)

        pred = predict_gamma_quiet(
            X_counts,
            Y_counts,
            rho_p
        )

        records.append({
            "sigma": sigma,
            "rep": rep,
            "alpha": pred["alpha"],
            "beta": pred["beta"]
        })

range_df = pd.DataFrame(records)

print("Total model runs:", len(range_df))
display(range_df.head())

## 4. Calculate empirical 95% ranges

For each output calculate:

- mean;
- standard deviation;
- 2.5th percentile;
- 97.5th percentile;
- range width;
- relative range width.

Relative width is useful because alpha and beta are on very different scales.

In [ ]:
def empirical_range_summary(df, value_col, baseline_value):
    rows = []

    for sigma, group in df.groupby("sigma"):
        lower = group[value_col].quantile(0.025)
        upper = group[value_col].quantile(0.975)
        width = upper - lower

        rows.append({
            "sigma": sigma,
            "baseline": baseline_value,
            "mean": group[value_col].mean(),
            "std": group[value_col].std(),
            "lower_2.5%": lower,
            "upper_97.5%": upper,
            "range_width": width,
            "relative_width_%": 100 * width / baseline_value
        })

    return pd.DataFrame(rows)


alpha_summary = empirical_range_summary(
    range_df, "alpha", baseline_alpha
)

beta_summary = empirical_range_summary(
    range_df, "beta", baseline_beta
)

print("Alpha / shape ranges")
display(alpha_summary.round(4))

print("Beta / rate ranges")
display(beta_summary.round(4))

## 5. Combined results table

This is the main table to use in your sprint review or report.

In [ ]:
alpha_small = alpha_summary[
    ["sigma", "baseline", "mean", "lower_2.5%", "upper_97.5%", "relative_width_%"]
].rename(columns={
    "baseline": "alpha_baseline",
    "mean": "alpha_mean",
    "lower_2.5%": "alpha_lower",
    "upper_97.5%": "alpha_upper",
    "relative_width_%": "alpha_relative_width_%"
})

beta_small = beta_summary[
    ["sigma", "baseline", "mean", "lower_2.5%", "upper_97.5%", "relative_width_%"]
].rename(columns={
    "baseline": "beta_baseline",
    "mean": "beta_mean",
    "lower_2.5%": "beta_lower",
    "upper_97.5%": "beta_upper",
    "relative_width_%": "beta_relative_width_%"
})

combined_summary = alpha_small.merge(beta_small, on="sigma")

display(combined_summary.round(4))

## 6. Plot alpha range

In [ ]:
plt.figure(figsize=(7, 4))

plt.errorbar(
    alpha_summary["sigma"],
    alpha_summary["mean"],
    yerr=[
        alpha_summary["mean"] - alpha_summary["lower_2.5%"],
        alpha_summary["upper_97.5%"] - alpha_summary["mean"]
    ],
    marker="o",
    capsize=4
)

plt.axhline(
    baseline_alpha,
    linestyle="--",
    label="Baseline alpha"
)

plt.xlabel("Perturbation sigma")
plt.ylabel("Predicted alpha / shape")
plt.title("Empirical 95% sensitivity range — alpha")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 7. Plot beta range

In [ ]:
plt.figure(figsize=(7, 4))

plt.errorbar(
    beta_summary["sigma"],
    beta_summary["mean"],
    yerr=[
        beta_summary["mean"] - beta_summary["lower_2.5%"],
        beta_summary["upper_97.5%"] - beta_summary["mean"]
    ],
    marker="o",
    capsize=4
)

plt.axhline(
    baseline_beta,
    linestyle="--",
    label="Baseline beta"
)

plt.xlabel("Perturbation sigma")
plt.ylabel("Predicted beta / rate")
plt.title("Empirical 95% sensitivity range — beta")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 8. Interpretation

Suggested reporting wording:

> We repeatedly perturbed the unreliability matrix while keeping the observed response distributions fixed. For each perturbation level, we recorded GemFinder's two neural-network outputs, Gamma shape (`alpha`) and Gamma rate (`beta`). Empirical 95% sensitivity ranges were then calculated using the 2.5th and 97.5th percentiles of the repeated predictions.

Interpret the results as follows:

- **narrow range** → output is relatively stable;
- **wide range** → output is more sensitive to rho uncertainty;
- **range width increases with sigma** → larger rho perturbations produce greater model-output uncertainty.

Because alpha and beta are on different numerical scales, also compare the **relative range width (%)**.

## 9. Export results

In [ ]:
OUTPUT_PATH = "/content/gemfinder_output_95pct_ranges.csv"

combined_summary.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)